# OsteoClassAI v2 · 12 Detector de articulación, paso 3: Gornale recortado con el detector

Se recorta cada radiografía de Gornale con el detector (entrenado solo con DetKneeData de Chen, sin ninguna imagen de Gornale) a la escala de los recortes de entrenamiento (1,30 × ancho de la caja) y se gradúa con **B** (referencia) y **D** (demo), congelados.

En la misma corrida salen, para comparar con el mismo protocolo: relleno a cuadrado (principal), estirado, recorte manual por articulación y recalibración por mitades.

**Es exploratorio:** Gornale ya se usó para tomar decisiones (cuadernos 06 a 08), así que no es una validación independiente.

*Entorno de ejecución:* GPU T4. Luego *Ejecutar todas*.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
if os.path.isdir('/content/osteoclassai'):
    !cd /content/osteoclassai && git pull -q
else:
    !git clone -q https://github.com/walterandres23/osteoclassai.git /content/osteoclassai
%cd /content/osteoclassai/v2
!pip -q install -e . timm imagehash h5py opencv-python-headless

In [ ]:
# Datos: el zip trae dos .rar (uno por experto)
ZIP = '/content/drive/MyDrive/OsteoClassAI/datos_externos/Digital Knee X-ray Images.zip'
!mkdir -p /content/gornale && cd /content/gornale && unzip -q -o "$ZIP"
!which unrar > /dev/null || apt-get -qq install -y unrar > /dev/null
!cd "/content/gornale/Digital Knee X-ray Images/Knee X-ray Images" && for f in *.rar; do unrar x -o+ -idq "$f"; done
root = '/content/gornale/Digital Knee X-ray Images/Knee X-ray Images'
!find "$root" -name '*.png' | wc -l   # debe dar 3300

In [ ]:
RUNS = '/content/drive/MyDrive/OsteoClassAI/runs'
!python -m osteoclass.external --root "$root" --runs $RUNS/convnext_tiny_sord_288 $RUNS/convnext_tiny_sord_robust --exploratory --detector $RUNS/detector/best.pt --out outputs/external/gornale_detector
!cp -r outputs/external/gornale_detector /content/drive/MyDrive/OsteoClassAI/external/

## Revisión visual: recorte del detector (rojo) y recorte manual por articulación (azul)

In [ ]:
import sys; sys.path.insert(0, '/content/osteoclassai/v2/src')
import pandas as pd, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as pa
from osteoclass.external import load_gray, joint_box
from osteoclass.detect import crop_box
det = pd.read_csv('outputs/external/gornale_detector/detections_single.csv')
aud = pd.read_csv('outputs/external/gornale_detector/audit.csv')
det = det.merge(aud[['path', 'grade_I']], on='path')
pick = pd.concat([det[det.grade_I == g].sample(2, random_state=0) for g in range(5)])
fig, ax = plt.subplots(2, 5, figsize=(20, 6))
for a, (_, r) in zip(ax.T.flat, pick.iterrows()):
    im = load_gray(f'{root}/{r.path}'); a.imshow(im, cmap='gray'); a.axis('off'); a.set_title(f'KL {r.grade_I}')
    x, y, s = joint_box(im); a.add_patch(pa.Rectangle((x, y), s, s, fill=False, ec='deepskyblue', lw=1.5))
    if not np.isnan(r.cx):
        x1, y1, x2, y2 = crop_box(r.cx, r.cy, r.w)
        a.add_patch(pa.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec='red', lw=1.5, ls='--'))
plt.tight_layout(); plt.show()
print('sin detección:', det.cx.isna().sum(), 'de', len(det))